# 🌾 Guide Pratique : Prédiction des Récoltes au Burundi

## Guide Étape par Étape pour un Projet de Machine Learning Complet

Ce notebook vous guide à travers toutes les étapes d'un projet de science des données, de la préparation des données au déploiement en ligne.

**Objectif :** Prédire la qualité des récoltes au Burundi en utilisant trois modèles de classification (Arbre de Décision, Forêt Aléatoire, Régression Logistique).

---

## 📋 Étape 1 : Chargement du Dataset

In [ ]:
# Importer les bibliothèques essentielles
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Configuration matplotlib
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Charger le dataset
df = pd.read_csv('../data/agriculture_burundi.csv')

# Afficher les informations de base
print("📊 Forme du dataset:", df.shape)
print("\n📝 Premières lignes:")
print(df.head())
print("\n📋 Informations sur les colonnes:")
print(df.info())
print("\n📈 Statistiques descriptives:")
print(df.describe())

## 🔍 Étape 2 : Analyse Exploratoire (EDA)

In [ ]:
# Détecter les valeurs manquantes
print("🔴 Valeurs manquantes:")
print(df.isnull().sum())

# Visualiser les colonnes numériques et catégorielles
numerical_cols = df.select_dtypes(include=[np.number]).columns
categorical_cols = df.select_dtypes(include=['object']).columns

print(f"\n📊 Colonnes numériques ({len(numerical_cols)}): {list(numerical_cols)}")
print(f"🏷️  Colonnes catégorielles ({len(categorical_cols)}): {list(categorical_cols)}")

# Distribution des variables numériques
fig, axes = plt.subplots(len(numerical_cols), 2, figsize=(14, 4*len(numerical_cols)))
for idx, col in enumerate(numerical_cols):
    axes[idx, 0].hist(df[col], bins=30, edgecolor='black', alpha=0.7)
    axes[idx, 0].set_title(f'Distribution de {col}', fontsize=12, fontweight='bold')
    axes[idx, 0].set_xlabel(col)
    
    axes[idx, 1].boxplot(df[col])
    axes[idx, 1].set_title(f'Boxplot de {col}', fontsize=12, fontweight='bold')
    axes[idx, 1].set_ylabel(col)

plt.tight_layout()
plt.show()

# Matrice de corrélation
if len(numerical_cols) > 1:
    plt.figure(figsize=(10, 8))
    corr_matrix = df[numerical_cols].corr()
    sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0, 
                square=True, linewidths=1, cbar_kws={"shrink": 0.8})
    plt.title('Matrice de Corrélation', fontsize=14, fontweight='bold')
    plt.show()

# Distribution des variables catégorielles
fig, axes = plt.subplots(len(categorical_cols), 1, figsize=(12, 4*len(categorical_cols)))
if len(categorical_cols) == 1:
    axes = [axes]
    
for idx, col in enumerate(categorical_cols):
    df[col].value_counts().plot(kind='bar', ax=axes[idx], color='skyblue', edgecolor='black')
    axes[idx].set_title(f'Répartition de {col}', fontsize=12, fontweight='bold')
    axes[idx].set_xlabel(col)
    axes[idx].set_ylabel('Fréquence')
    axes[idx].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 🛠️ Étape 3 : Traitement des Valeurs Manquantes

In [ ]:
from sklearn.impute import SimpleImputer, KNNImputer

# Afficher les valeurs manquantes par colonne
missing_data = df.isnull().sum()
missing_percent = (missing_data / len(df)) * 100
missing_df = pd.DataFrame({
    'Colonne': missing_data.index,
    'Manquantes': missing_data.values,
    'Pourcentage': missing_percent.values
})
missing_df = missing_df[missing_df['Manquantes'] > 0].sort_values('Manquantes', ascending=False)

if len(missing_df) > 0:
    print("📊 Résumé des valeurs manquantes:")
    print(missing_df)
    
    # Stratégie 1: Supprimer les lignes avec trop de valeurs manquantes
    threshold = 0.5
    df_clean = df.dropna(thresh=len(df.columns) * threshold)
    print(f"\n✂️  Suppression des lignes avec > {threshold*100}% manquantes: {len(df)} → {len(df_clean)} lignes")
    
    # Stratégie 2: Imputer les colonnes numériques avec la médiane
    imputer_num = SimpleImputer(strategy='median')
    for col in numerical_cols:
        if df_clean[col].isnull().sum() > 0:
            df_clean[col] = imputer_num.fit_transform(df_clean[[col]])
    
    # Stratégie 3: Imputer les colonnes catégorielles avec le mode
    imputer_cat = SimpleImputer(strategy='most_frequent')
    for col in categorical_cols:
        if df_clean[col].isnull().sum() > 0:
            df_clean[col] = imputer_cat.fit_transform(df_clean[[col]])
    
    df = df_clean
    print(f"\n✅ Imputation terminée. Valeurs manquantes restantes: {df.isnull().sum().sum()}")
else:
    print("✅ Aucune valeur manquante détectée!")

## 🔤 Étape 4 : Encodage des Variables Catégorielles

In [ ]:
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, ColumnTransformer, Pipeline
from sklearn.compose import ColumnTransformer

# Afficher les variables catégorielles
print(f"🏷️  Variables catégorielles: {list(categorical_cols)}")

# Créer un encodeur OneHot pour les variables nominales
encoder = OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore')

# Appliquer l'encodage
df_encoded = df.copy()

for col in categorical_cols:
    encoded = encoder.fit_transform(df_encoded[[col]])
    encoded_df = pd.DataFrame(
        encoded,
        columns=encoder.get_feature_names_out([col])
    )
    df_encoded = pd.concat([df_encoded.drop(col, axis=1), encoded_df], axis=1)

print(f"\n📊 Nouvelles dimensions après encodage: {df_encoded.shape}")
print(f"\n✅ Colonnes après encodage:")
print(df_encoded.columns.tolist())

## 📏 Étape 5 : Normalisation / Mise à l'Échelle des Données

In [ ]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler

# Utiliser les données encodées
X = df_encoded.drop('yield', axis=1) if 'yield' in df_encoded.columns else df_encoded.iloc[:, :-1]
y = df_encoded['yield'] if 'yield' in df_encoded.columns else df_encoded.iloc[:, -1]

# StandardScaler : centrer et réduire (moyenne 0, écart-type 1)
scaler_standard = StandardScaler()
X_scaled_standard = scaler_standard.fit_transform(X)

# MinMaxScaler : ramener à [0, 1]
scaler_minmax = MinMaxScaler()
X_scaled_minmax = scaler_minmax.fit_transform(X)

print("📊 Normalisation StandardScaler:")
print(f"  Moyenne: {X_scaled_standard.mean():.4f}")
print(f"  Écart-type: {X_scaled_standard.std():.4f}")
print(f"  Min: {X_scaled_standard.min():.4f}, Max: {X_scaled_standard.max():.4f}")

print("\n📊 Normalisation MinMaxScaler:")
print(f"  Min: {X_scaled_minmax.min():.4f}, Max: {X_scaled_minmax.max():.4f}")

# Utiliser StandardScaler pour la suite (plus courant)
X_scaled = X_scaled_standard
X_scaled = pd.DataFrame(X_scaled, columns=X.columns)

print("\n✅ Données normalisées prêtes!")

## ✂️ Étape 6 : Division Train / Test

In [ ]:
from sklearn.model_selection import train_test_split

# Diviser les données en 80% train et 20% test
test_size = 0.2
random_state = 42

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y,
    test_size=test_size,
    random_state=random_state,
    stratify=y if y.dtype == 'object' or len(y.unique()) < 10 else None
)

print(f"📊 Division Train/Test:")
print(f"  Ensemble d'entraînement: {X_train.shape[0]} samples ({100*(1-test_size):.0f}%)")
print(f"  Ensemble de test: {X_test.shape[0]} samples ({100*test_size:.0f}%)")
print(f"  Nombre de features: {X_train.shape[1]}")

if hasattr(y, 'dtype') and (y.dtype == 'object' or len(y.unique()) < 10):
    print(f"\n📈 Distribution des classes en train: {pd.Series(y_train).value_counts().to_dict()}")
    print(f"📈 Distribution des classes en test: {pd.Series(y_test).value_counts().to_dict()}")

## 🌳 Étape 7 : Entraînement d'un Arbre de Décision

In [ ]:
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.model_selection import cross_val_score

# Créer et entraîner l'Arbre de Décision
dt_model = DecisionTreeClassifier(
    max_depth=10,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42
)

dt_model.fit(X_train, y_train)

# Prédictions
y_pred_dt = dt_model.predict(X_test)
y_pred_proba_dt = dt_model.predict_proba(X_test)

# Validation croisée
cv_scores = cross_val_score(dt_model, X_train, y_train, cv=5)

print("🌳 Arbre de Décision:")
print(f"  Profondeur maximale: {dt_model.get_depth()}")
print(f"  Nombre de feuilles: {dt_model.get_n_leaves()}")
print(f"  Score de validation croisée (5-fold): {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")

# Visualiser les importances des features
feat_importance = pd.DataFrame({
    'Feature': X_train.columns,
    'Importance': dt_model.feature_importances_
}).sort_values('Importance', ascending=False).head(10)

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(feat_importance['Feature'], feat_importance['Importance'], color='steelblue')
ax.set_xlabel('Importance', fontweight='bold')
ax.set_title('Top 10 Features - Arbre de Décision', fontweight='bold')
plt.tight_layout()
plt.show()

print("\n📊 Top 10 Features:")
print(feat_importance)

## 🌲 Étape 8 : Entraînement d'une Forêt Aléatoire

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV

# Créer la Forêt Aléatoire
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=15,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

# Prédictions
y_pred_rf = rf_model.predict(X_test)
y_pred_proba_rf = rf_model.predict_proba(X_test)

# Validation croisée
cv_scores_rf = cross_val_score(rf_model, X_train, y_train, cv=5)

print("🌲 Forêt Aléatoire:")
print(f"  Nombre d'arbres: {rf_model.n_estimators}")
print(f"  Score de validation croisée (5-fold): {cv_scores_rf.mean():.4f} (+/- {cv_scores_rf.std():.4f})")

# Tuning des hyperparamètres (optionnel - peut prendre du temps)
# param_grid = {
#     'n_estimators': [50, 100, 150],
#     'max_depth': [10, 15, 20]
# }
# grid_search = GridSearchCV(RandomForestClassifier(random_state=42), param_grid, cv=5, n_jobs=-1)
# grid_search.fit(X_train, y_train)
# print(f"Meilleurs paramètres: {grid_search.best_params_}")

# Importances des features
feat_importance_rf = pd.DataFrame({
    'Feature': X_train.columns,
    'Importance': rf_model.feature_importances_
}).sort_values('Importance', ascending=False).head(10)

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(feat_importance_rf['Feature'], feat_importance_rf['Importance'], color='seagreen')
ax.set_xlabel('Importance', fontweight='bold')
ax.set_title('Top 10 Features - Forêt Aléatoire', fontweight='bold')
plt.tight_layout()
plt.show()

print("\n📊 Top 10 Features:")
print(feat_importance_rf)

## 📈 Étape 9 : Entraînement d'une Régression Logistique

In [ ]:
from sklearn.linear_model import LogisticRegression

# Créer et entraîner la Régression Logistique
lr_model = LogisticRegression(
    max_iter=1000,
    random_state=42,
    solver='lbfgs',
    multi_class='multinomial'
)

lr_model.fit(X_train, y_train)

# Prédictions
y_pred_lr = lr_model.predict(X_test)
y_pred_proba_lr = lr_model.predict_proba(X_test)

# Validation croisée
cv_scores_lr = cross_val_score(lr_model, X_train, y_train, cv=5)

print("📈 Régression Logistique:")
print(f"  Nombre d'itérations: {lr_model.n_iter_[0] if hasattr(lr_model, 'n_iter_') else 'N/A'}")
print(f"  Score de validation croisée (5-fold): {cv_scores_lr.mean():.4f} (+/- {cv_scores_lr.std():.4f})")

# Afficher les coefficients
coef_df = pd.DataFrame({
    'Feature': X_train.columns,
    'Coefficient': lr_model.coef_[0] if lr_model.coef_.shape[0] == 1 else lr_model.coef_[0]
}).sort_values('Coefficient', key=abs, ascending=False).head(10)

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(coef_df['Feature'], coef_df['Coefficient'], color='coral')
ax.set_xlabel('Coefficient', fontweight='bold')
ax.set_title('Top 10 Features - Régression Logistique', fontweight='bold')
plt.tight_layout()
plt.show()

print("\n📊 Top 10 Features:")
print(coef_df)

## 🎯 Étape 10 : Évaluation des Modèles

In [ ]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)

# Évaluation des trois modèles
models = {
    'Arbre de Décision': (y_pred_dt, dt_model),
    'Forêt Aléatoire': (y_pred_rf, rf_model),
    'Régression Logistique': (y_pred_lr, lr_model)
}

results = []

for model_name, (y_pred, model) in models.items():
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average='weighted')
    rec = recall_score(y_test, y_pred, average='weighted')
    f1 = f1_score(y_test, y_pred, average='weighted')
    
    results.append({
        'Modèle': model_name,
        'Accuracy': acc,
        'Précision': prec,
        'Rappel': rec,
        'F1-Score': f1
    })
    
    print(f"\n📊 {model_name}")
    print(f"  Accuracy: {acc:.4f}")
    print(f"  Précision: {prec:.4f}")
    print(f"  Rappel: {rec:.4f}")
    print(f"  F1-Score: {f1:.4f}")
    print(f"\nMatrice de confusion:\n{confusion_matrix(y_test, y_pred)}")

# Créer un DataFrame de comparaison
results_df = pd.DataFrame(results).set_index('Modèle')
print("\n\n🏆 Comparaison des modèles:")
print(results_df)

# Visualiser la comparaison
fig, ax = plt.subplots(figsize=(12, 6))
results_df.plot(kind='bar', ax=ax, width=0.8)
ax.set_title('Comparaison des Modèles', fontsize=14, fontweight='bold')
ax.set_ylabel('Score', fontweight='bold')
ax.set_xlabel('')
ax.legend(title='Métrique', bbox_to_anchor=(1.05, 1), loc='upper left')
ax.set_ylim([0, 1.05])
plt.tight_layout()
plt.show()

## 📉 Étape 11 : Courbe ROC et AUC

In [ ]:
from sklearn.metrics import roc_curve, auc, roc_auc_score
from sklearn.preprocessing import label_binarize

# Déterminer si problème binaire ou multiclasse
n_classes = len(np.unique(y_test))
is_binary = n_classes == 2

# Tracer les courbes ROC
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

model_proba_list = [
    ('Arbre de Décision', y_pred_proba_dt),
    ('Forêt Aléatoire', y_pred_proba_rf),
    ('Régression Logistique', y_pred_proba_lr)
]

for idx, (model_name, y_pred_proba) in enumerate(model_proba_list):
    ax = axes[idx]
    
    if is_binary:
        # Cas binaire
        fpr, tpr, _ = roc_curve(y_test, y_pred_proba[:, 1])
        roc_auc = auc(fpr, tpr)
        ax.plot(fpr, tpr, 'b-', linewidth=2, label=f'ROC (AUC = {roc_auc:.3f})')
    else:
        # Cas multiclasse: One-vs-Rest
        y_test_bin = label_binarize(y_test, classes=np.unique(y_test))
        for i in range(n_classes):
            fpr, tpr, _ = roc_curve(y_test_bin[:, i], y_pred_proba[:, i])
            roc_auc = auc(fpr, tpr)
            ax.plot(fpr, tpr, linewidth=2, label=f'Classe {i} (AUC = {roc_auc:.3f})')
    
    ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Aléatoire')
    ax.set_xlabel('Taux de Faux Positifs', fontweight='bold')
    ax.set_ylabel('Taux de Vrais Positifs', fontweight='bold')
    ax.set_title(f'Courbe ROC - {model_name}', fontweight='bold')
    ax.legend(loc='lower right')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Calculer l'AUC pour chaque modèle
print("\n📊 Scores AUC:")
for model_name, y_pred_proba in model_proba_list:
    if is_binary:
        auc_score = roc_auc_score(y_test, y_pred_proba[:, 1])
        print(f"  {model_name}: {auc_score:.4f}")
    else:
        auc_score = roc_auc_score(y_test, y_pred_proba, multi_class='ovr', average='weighted')
        print(f"  {model_name}: {auc_score:.4f} (weighted)")

## 💾 Étape 12 : Sauvegarde des Modèles (.pkl)

In [ ]:
import joblib
import pickle
from datetime import datetime

# Créer un pipeline complet pour chaque modèle (preprocessing + modèle)
pipeline_dt = Pipeline([
    ('scaler', StandardScaler()),
    ('model', dt_model)
])

pipeline_rf = Pipeline([
    ('scaler', StandardScaler()),
    ('model', rf_model)
])

pipeline_lr = Pipeline([
    ('scaler', StandardScaler()),
    ('model', lr_model)
])

# Créer le dossier models s'il n'existe pas
import os
os.makedirs('../models', exist_ok=True)

# Sauvegarder les modèles avec timestamp
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

models_to_save = {
    'decision_tree': dt_model,
    'random_forest': rf_model,
    'logistic_regression': lr_model
}

for model_name, model in models_to_save.items():
    filepath = f'../models/{model_name}_{timestamp}.pkl'
    joblib.dump(model, filepath)
    print(f"✅ {model_name} sauvegardé: {filepath}")

# Sauvegarder aussi les transformateurs
joblib.dump(scaler_standard, f'../models/scaler_{timestamp}.pkl')
print(f"✅ Scaler sauvegardé: ../models/scaler_{timestamp}.pkl")

# Sauvegarder les métriques dans un JSON
import json

metrics = {
    'timestamp': timestamp,
    'models': {}
}

for model_name, (y_pred, model) in models.items():
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average='weighted')
    rec = recall_score(y_test, y_pred, average='weighted')
    f1 = f1_score(y_test, y_pred, average='weighted')
    
    metrics['models'][model_name] = {
        'accuracy': float(acc),
        'precision': float(prec),
        'recall': float(rec),
        'f1_score': float(f1)
    }

with open(f'../models/metrics_{timestamp}.json', 'w') as f:
    json.dump(metrics, f, indent=4)
    print(f"✅ Métriques sauvegardées: ../models/metrics_{timestamp}.json")

print("\n📦 Tous les modèles ont été sauvegardés!")

## 🌐 Étape 13 : Développement d'une Application Web Streamlit

Pour créer l'application Streamlit, créez un fichier `app.py` dans le dossier `app/` avec le contenu suivant:

In [ ]:
# Afficher le code exemple pour l'application Streamlit
streamlit_code = """
import streamlit as st
import pandas as pd
import numpy as np
import joblib
import json
from datetime import datetime

# Configuration
st.set_page_config(
    page_title="🌾 AgriPredict Burundi",
    layout="wide",
    initial_sidebar_state="expanded"
)

st.title("🌾 Prédiction des Récoltes au Burundi")

# Charger les modèles
try:
    model_rf = joblib.load('../models/random_forest_*.pkl')
    scaler = joblib.load('../models/scaler_*.pkl')
except:
    st.error("Erreur: Modèles non trouvés. Veuillez d'abord entraîner les modèles.")

# Sidebar
st.sidebar.header("⚙️ Paramètres")

altitude = st.sidebar.slider("⛰️ Altitude (m)", 400, 2600, 1720)
pluviometrie = st.sidebar.slider("🌧️ Pluviométrie (mm)", 150, 1600, 900)
temperature = st.sidebar.slider("🌡️ Température (°C)", 14.0, 30.0, 18.2)
superficie = st.sidebar.slider("📐 Superficie (ha)", 5.0, 800.0, 50.0)
engrais = st.sidebar.checkbox("🧪 Utilisation d'engrais", value=True)

# Prédiction
if st.sidebar.button("🔍 PRÉDIRE"):
    # Préparer les données
    input_data = np.array([[altitude, pluviometrie, temperature, superficie, int(engrais)]])
    input_scaled = scaler.transform(input_data)
    
    # Faire la prédiction
    prediction = model_rf.predict(input_scaled)
    probability = model_rf.predict_proba(input_scaled).max()
    
    # Afficher les résultats
    if prediction[0] == 1:
        st.success(f"✅ **BONNE RÉCOLTE** ({probability*100:.1f}% de confiance)")
    else:
        st.warning(f"⚠️ **MAUVAISE RÉCOLTE** ({probability*100:.1f}% de confiance)")
    
    # Afficher les paramètres
    col1, col2 = st.columns(2)
    with col1:
        st.metric("Altitude", f"{altitude} m")
        st.metric("Température", f"{temperature} °C")
    with col2:
        st.metric("Pluviométrie", f"{pluviometrie} mm")
        st.metric("Engrais utilisé", "Oui" if engrais else "Non")
"""

print("📝 Code exemple pour Streamlit App:")
print(streamlit_code)

# Sauvegarder dans un fichier
with open('../app/example_app.py', 'w', encoding='utf-8') as f:
    f.write(streamlit_code)

print("\n✅ Code Streamlit d'exemple sauvegardé dans ../app/example_app.py")

## 🚀 Étape 14 : Déploiement en Ligne

In [ ]:
# Créer les fichiers nécessaires au déploiement

# 1. requirements.txt
requirements = """pandas==2.0.3
numpy==1.24.3
scikit-learn==1.3.0
matplotlib==3.7.2
seaborn==0.12.2
plotly==5.16.1
streamlit==1.28.0
joblib==1.3.1
"""

with open('../requirements.txt', 'w') as f:
    f.write(requirements)
print("✅ requirements.txt créé")

# 2. .gitignore
gitignore = """
__pycache__/
*.pyc
*.pkl
*.json
.DS_Store
.env
venv/
.streamlit/
"""

with open('../.gitignore', 'w') as f:
    f.write(gitignore)
print("✅ .gitignore créé")

# 3. Dockerfile (optionnel pour déploiement Docker)
dockerfile = """FROM python:3.11-slim

WORKDIR /app

COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

COPY . .

EXPOSE 8501

CMD ["streamlit", "run", "app/app.py"]
"""

with open('../Dockerfile', 'w') as f:
    f.write(dockerfile)
print("✅ Dockerfile créé")

# 4. Configuration Streamlit
streamlit_config = """[theme]
primaryColor = "#FF6B6B"
backgroundColor = "#F0F2F6"
secondaryBackgroundColor = "#E8EAED"
textColor = "#262730"

[client]
maxUploadSize = 200
"""

os.makedirs('../.streamlit', exist_ok=True)
with open('../.streamlit/config.toml', 'w') as f:
    f.write(streamlit_config)
print("✅ Configuration Streamlit créée")

print("""
📋 Checklist de Déploiement:

1. ✅ requirements.txt - Dépendances Python
2. ✅ .gitignore - Fichiers à ignorer
3. ✅ Dockerfile - Pour déploiement Docker
4. ✅ .streamlit/config.toml - Configuration Streamlit

🌐 Options de Déploiement:

A. STREAMLIT CLOUD (Recommandé - Plus facile)
   1. Créer un compte GitHub
   2. Pousser votre code sur GitHub
   3. Aller sur https://streamlit.io/cloud
   4. Sélectionner votre repo et déployer
   
B. DOCKER + HEROKU
   1. Installer Docker
   2. Créer un compte Heroku
   3. heroku login
   4. heroku create app-name
   5. git push heroku main
   
C. RAILWAY / RENDER
   1. Connecter votre repo GitHub
   2. Configurer les variables d'environnement
   3. Déployer automatiquement

📝 Variables d'Environnement à configurer:
   - MODEL_PATH: Chemin vers le modèle .pkl
   - SCALER_PATH: Chemin vers le scaler
   - DEBUG: True/False pour le mode debug
""")

## 📚 Conclusion et Prochaines Étapes

### 🎯 Récapitulatif du TP

Vous avez complété un cycle complet de **Machine Learning en Production**:

1. ✅ **Préparation des données** : Chargement, nettoyage, encodage, normalisation
2. ✅ **Modélisation** : Trois modèles entraînés (Arbre, Forêt, Régression Logistique)
3. ✅ **Évaluation** : Métriques complètes (Accuracy, F1, AUC-ROC)
4. ✅ **Sauvegarde** : Modèles persistants en .pkl
5. ✅ **Application Web** : Interface utilisateur Streamlit
6. ✅ **Déploiement** : Prêt pour production

### 🚀 Améliorations Futures

- **Hyperparameter Tuning** : GridSearchCV pour optimiser chaque modèle
- **Ensemble Methods** : Stacking, Voting pour combiner modèles
- **Feature Engineering** : Créer nouvelles variables pertinentes
- **Time Series** : Analyser tendances temporelles si données disponibles
- **Monitoring** : Suivre la performance en production
- **A/B Testing** : Comparer modèles en production

### 📞 Support et Ressources

- **Documentation Scikit-learn** : https://scikit-learn.org/
- **Streamlit Docs** : https://docs.streamlit.io/
- **Kaggle Datasets** : https://www.kaggle.com/datasets
- **Machine Learning Mastery** : https://machinelearningmastery.com/

---

**Félicitations! 🎉 Vous avez complété le TP "Prédiction des Récoltes au Burundi"**